# TikTok Research API – Sampling
Query 400 TikTok videos with #adhd, #adhdtiktok, #adhdlife
Time window: November 2025 - May 2026 | Region: US, GB, AU, CA

In [ ]:
# ============================================================
# Load API Credentials
# ============================================================
from dotenv import load_dotenv                                                                       
import os                                                                                            
                                                                                                    
load_dotenv("/Users/baiyun/Documents/Bachelorarbeit/code/.env")                                      
                                                                                                       
CLIENT_KEY = os.getenv("CLIENT_KEY")                                                                 
CLIENT_SECRET = os.getenv("CLIENT_SECRET")
# ============================================================

In [ ]:
# ============================================================
# Import Packages 
# ============================================================
import requests
import pandas as pd
from datetime import datetime

print("Packages loaded.")

In [ ]:
# ============================================================
# Authenticate with TikTok Research API
# ============================================================
def get_access_token(client_key, client_secret):
    url = "https://open.tiktokapis.com/v2/oauth/token/"
    headers = {"Content-Type": "application/x-www-form-urlencoded"}
    data = {
        "client_key": client_key,
        "client_secret": client_secret,
        "grant_type": "client_credentials"
    }
    response = requests.post(url, headers=headers, data=data)
    response.raise_for_status()
    token = response.json()["access_token"]
    print("Access token successfully obtained.")
    return token

ACCESS_TOKEN = get_access_token(CLIENT_KEY, CLIENT_SECRET)

In [ ]:
# ============================================================
# Query Videos: Hashtag_based Sampling via TikTok Research API 
# ============================================================

TARGET = 300

def query_videos_range(access_token, start, end, max_count=100):
    url = "https://open.tiktokapis.com/v2/research/video/query/"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }
    fields = (
        "id,video_description,create_time,username,"
        "region_code,view_count,like_count,comment_count,"
        "share_count,voice_to_text"
    )
    all_videos = []
    cursor = 0
    search_id = None
    has_more = True

    # Pagination: keep requesting until no more videos available
    while has_more:
        body = {
            "query": {
                "or": [
                    {"operation": "EQ", "field_name": "hashtag_name", "field_values": ["adhd"]},
                    {"operation": "EQ", "field_name": "hashtag_name", "field_values": ["adhdtiktok"]},
                    {"operation": "EQ", "field_name": "hashtag_name", "field_values": ["adhdlife"]}
                ],
                "and": [
                    {"operation": "IN", "field_name": "region_code", "field_values": ["US", "GB", "CA", "AU"]}
                ]
            },
            "max_count": max_count,
            "cursor": cursor,
            "start_date": start,
            "end_date": end
        }
        if search_id:
            body["search_id"] = search_id

        response = requests.post(
            url,
            headers=headers,
            params={"fields": fields},
            json=body
        )
        if response.status_code != 200:
            print(f"Error {response.status_code}: {response.text}")
            break
        data = response.json().get("data", {})
        all_videos.extend(data.get("videos", []))
        has_more = data.get("has_more", False)
        cursor = data.get("cursor", 0)
        search_id = data.get("search_id", search_id)

    return all_videos


# 6 months: 26 November 2025 - 26 May 2026
blocks = [
    ("20251126", "20251130"),
    ("20251201", "20251214"),
    ("20251215", "20251231"),
    ("20260101", "20260114"),
    ("20260115", "20260131"),
    ("20260201", "20260214"),
    ("20260215", "20260228"),
    ("20260301", "20260314"),
    ("20260315", "20260331"),
    ("20260401", "20260414"),
    ("20260415", "20260430"),
    ("20260501", "20260514"),
    ("20260515", "20260526"),
]

all_videos = []
seen_ids = set()

for start, end in blocks:
    if len(all_videos) >= TARGET:
        print(f"Target of {TARGET} videos reached - stopping.")
        break
    result = query_videos_range(ACCESS_TOKEN, start, end)
    for v in result:
        if int(v.get("view_count", 0)) >= 10000 and v["id"] not in seen_ids:
            seen_ids.add(v["id"])
            all_videos.append(v)
            if len(all_videos) >= TARGET:
                break
    print(f"{start} to {end}: {len(result)} API videos -> {len(all_videos)} accepted")

print(f"\nTotal: {len(all_videos)} videos")
videos = all_videos


In [ ]:
# ============================================================
# Process and Previes Results
# ============================================================
if videos:
    df = pd.DataFrame(videos)
    if "create_time" in df.columns:
        df["create_time"] = pd.to_datetime(df["create_time"], unit="s")
    if "id" in df.columns and "username" in df.columns:
        df["video_url"] = "https://www.tiktok.com/@" + df["username"] + "/video/" + df["id"].astype(str)
    print(f"Columns: {list(df.columns)}")
    print(f"Number of videos: {len(df)}")
    print(df["region_code"].value_counts())
    display(df)
else:
    print("No videos received.")

In [ ]:
# ============================================================
# Export Results to CSV and Markdown
# ============================================================
if videos:
    # 1. Raw CSV – all fields
    df.to_csv("tiktok_pretest_sample.csv", index=False, encoding="utf-8-sig")
    print("Saved: tiktok_pretest_sample.csv")

    # 2. Polished CSV – without long text fields
    polished = df[[
        "id", "video_url", "username", "create_time",
        "region_code", "view_count", "like_count",
        "comment_count", "share_count"
    ]].copy()
    polished.to_csv("tiktok_sample_polished.csv", index=False, encoding="utf-8-sig")
    print("Saved: tiktok_sample_polished.csv")

    # 3. Markdown – clickable links
    lines = []
    lines.append(f"# TikTok ADHD Sample – {len(polished)} Videos\n")
    lines.append("| # | Username | Region | Date | Views | Likes | Comments | Shares | Link |")
    lines.append("|---|---|---|---|---|---|---|---|---|")
    for i, row in polished.iterrows():
        date = str(row["create_time"])[:10]
        link = f"[open]({row['video_url']})"
        lines.append(
            f"| {i+1} | {row['username']} | {row['region_code']} | {date} | "
            f"{int(row['view_count']):,} | {int(row['like_count']):,} | "
            f"{int(row['comment_count']):,} | {int(row['share_count']):,} | {link} |"
        )
    with open("sample_polished.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print("Saved: sample_polished.md")


In [ ]:
# ============================================================
# Anonymization: Remove usernames, video URLs, and IDs
# ============================================================
import pandas as pd

df_anon = pd.read_csv("../data/final_coding_sheet.csv")

# Replace usernames with anonymized IDs
usernames = df_anon["username"].unique()
mapping = {name: f"creator_{i+1}" for i, name in enumerate(usernames)}
df_anon["username"] = df_anon["username"].map(mapping)

# Remove video URLs if present
if "video_url" in df_anon.columns:
    df_anon = df_anon.drop(columns=["video_url"])

# Remove video IDs if present
if "id" in df_anon.columns:
    df_anon = df_anon.drop(columns=["id"])

print(f"Anonymized {len(usernames)} unique usernames.")
print(f"Columns after anonymization: {list(df_anon.columns)}")
print(f"\nFirst 5 rows:")
display(df_anon.head())